# Ranking Chicago Attractions by Wikipedia Pageviews

The OSM attractions feed has no notion of importance: the Bean and a small
neighborhood gallery are equal rows. This notebook attaches a popularity score
to each attraction using Wikipedia pageviews, which serve as a public,
keyless proxy for visitor interest.

**Pipeline:**
1. Pull tourist attractions from the Overpass API (same query the app uses).
2. Resolve each attraction to an English Wikipedia article. Primary source is
   the `wikipedia` tag on the OSM element itself. Fallback is Wikipedia's
   geosearch API within 300m, accepted only when the article title is a close
   string match to the attraction name.
3. Pull monthly pageviews for the last 12 complete months from the Wikimedia
   REST API and average them.
4. Export `data/attractions.csv`. The app prefers this file over a live
   Overpass call and unlocks the popularity filter and cluster ranking.

Pageviews measure article interest, not foot traffic. The two correlate well
at the top (Millennium Park, Willis Tower) but a place can be famous to read
about and mediocre to visit. The score is used for ranking and filtering,
never presented as visitor counts.

In [ ]:
import io
import json
import sys
import time
import urllib.request
import urllib.parse
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from difflib import SequenceMatcher

sys.path.append("..")
from helpers import OVERPASS_ENDPOINTS, OVERPASS_QUERY, _osm_category, _wiki_title, _display_name

HEADERS = {"User-Agent": "chicago-tourist-transport-guide (github.com/lewilliam888)"}
OUT_PATH = "../data/attractions.csv"


def get_json(url, data=None):
    req = urllib.request.Request(url, data=data, headers=HEADERS)
    with urllib.request.urlopen(req, timeout=120) as resp:
        return json.loads(resp.read().decode())

## 1. Attractions from Overpass

In [ ]:
payload = None
for endpoint in OVERPASS_ENDPOINTS:
    try:
        payload = get_json(endpoint, data=urllib.parse.urlencode({"data": OVERPASS_QUERY}).encode())
        break
    except Exception as e:
        print(f"{endpoint} failed: {e}")
assert payload is not None, "all Overpass servers busy, retry in a minute"

rows = []
for el in payload.get("elements", []):
    tags = el.get("tags", {})
    name = tags.get("name")
    category = _osm_category(tags)
    if not name or category is None:
        continue
    lat = el.get("lat") or el.get("center", {}).get("lat")
    lon = el.get("lon") or el.get("center", {}).get("lon")
    if lat is None or lon is None:
        continue
    rows.append({
        "name": _display_name(tags),
        "category": category,
        "latitude": float(lat),
        "longitude": float(lon),
        "wiki_title": _wiki_title(tags),
        "wikidata": tags.get("wikidata"),
    })

df = pd.DataFrame(rows)
has_wd = df["wikidata"].notna()
df = pd.concat([df[has_wd].drop_duplicates("wikidata"), df[~has_wd]])
df = df.drop_duplicates(["name", "category"]).reset_index(drop=True)
print(f"{len(df)} attractions")
df["category"].value_counts()

## 2. Resolve Wikipedia articles

The `wikipedia` tag covers the well-mapped attractions. For the rest, search
articles within 300m of the coordinates and accept the best title only when
its string similarity to the attraction name is at least 0.6. That threshold
keeps "Millennium Park" matching "Millennium Park" while rejecting an
adjacent skyscraper's article.

In [ ]:
def geosearch_match(name, lat, lon, radius=300):
    url = ("https://en.wikipedia.org/w/api.php?action=query&list=geosearch"
           f"&gscoord={lat}%7C{lon}&gsradius={radius}&gslimit=5&format=json")
    try:
        result = get_json(url)
        candidates = result.get("query", {}).get("geosearch", [])
    except Exception:
        return None
    best, best_score = None, 0.0
    for c in candidates:
        score = SequenceMatcher(None, name.lower(), c["title"].lower()).ratio()
        if score > best_score:
            best, best_score = c["title"], score
    return best if best_score >= 0.6 else None


missing = df["wiki_title"].isna()
print(f"{(~missing).sum()} titles from OSM tags, resolving {missing.sum()} via geosearch...")
for i in df.index[missing]:
    df.loc[i, "wiki_title"] = geosearch_match(
        df.loc[i, "name"], df.loc[i, "latitude"], df.loc[i, "longitude"])
    time.sleep(0.05)

print(f"matched: {df['wiki_title'].notna().sum()} of {len(df)}")

## 3. Pageviews

Monthly views for the last 12 complete months, averaged. Articles without
pageview data (404s) keep NaN and sort last in the app.

In [ ]:
end = date.today().replace(day=1)                      # first of current month
start = end.replace(year=end.year - 1)                 # 12 months back
fmt = lambda d: d.strftime("%Y%m%d") + "00"


def monthly_views(title):
    quoted = urllib.parse.quote(title.replace(" ", "_"), safe="")
    url = ("https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/"
           f"en.wikipedia/all-access/user/{quoted}/monthly/{fmt(start)}/{fmt(end)}")
    try:
        items = get_json(url).get("items", [])
        return float(np.mean([x["views"] for x in items])) if items else np.nan
    except Exception:
        return np.nan


titled = df["wiki_title"].notna()
print(f"fetching pageviews for {titled.sum()} articles...")
views = {}
for title in df.loc[titled, "wiki_title"].unique():
    views[title] = monthly_views(title)
    time.sleep(0.05)

df["monthly_views"] = df["wiki_title"].map(views)
print(f"got views for {df['monthly_views'].notna().sum()} attractions")

## 4. Sanity check: does the ranking look like Chicago?

In [ ]:
top = df.nlargest(20, "monthly_views")

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top["name"][::-1], top["monthly_views"][::-1])
ax.set_xlabel("Wikipedia views per month (12-month avg)")
ax.set_title("Top 20 Chicago attractions by Wikipedia pageviews")
plt.tight_layout()
plt.show()

top[["name", "category", "monthly_views"]].reset_index(drop=True)

In [ ]:
df.groupby("category")["monthly_views"].agg(
    attractions="size",
    with_views=lambda s: s.notna().sum(),
    median_views="median",
    max_views="max",
).round(0)

## 5. Export

In [ ]:
out = df[["name", "category", "latitude", "longitude", "wiki_title", "monthly_views"]]
out.to_csv(OUT_PATH, index=False)
print(f"wrote {len(out)} rows to {OUT_PATH}")

## Limitations

- Pageviews measure reading interest, not visits. Sports venues and buildings
  with famous histories over-index; purely visual attractions under-index.
- Unmatched attractions (no OSM tag, no confident geosearch match) get NaN
  and are excluded from popularity filtering rather than guessed.
- OSM coverage and tagging vary by neighborhood. A missing attraction here is
  missing from the app.
- The 0.6 similarity threshold trades recall for precision. Spot-check the
  matches before trusting a new export.